#### mock code flow
- [x] func extract_output_results (file_1, file_2) -> grades_1, grades_2
- [x] func calc_test_statistics (grades_1, grades_2) -> n12, n21, n_star, z0
- func do_benjamini_hochberg (z0) -> p_values, rejections


In [1]:
import json
import numpy as np
from scipy.stats import binomtest

In [2]:
import json
import numpy as np

In [3]:
file_1 = "outputs/claude-3-opus-20240229/linda_variant_one_to/responses_baseline_synthetic_dataset_linda_variant_one_to_gold.json"
file_2 = "outputs/claude-3-opus-20240229/linda_variant_one_to/responses_baseline_synthetic_dataset_linda_variant_one_to_random.json"

In [4]:
#func extract_output_results (file_1, file_2) -> grades_1, grades_2

def extract_output_results(file_path):
    # Note: not secured against edge cases (like if key "init_grades" is anything else than ["[Correct]"] or ["[Incorrect]"])
    with open(file_path, 'r') as f:
        data = json.load(f)
    length = data["stats"]["count_total"]
    grades = np.full(length, False)
    for i in range(length):
        grade = data[f"{i}"]["init_grades"][0]
        grades[i] = grade == "[Correct]"
    return grades

grades_1 = extract_output_results(file_1)
grades_2 = extract_output_results(file_2)

In [5]:
#func calc_test_statistics (grades_1, grades_2) -> n12, n21, n_star, z0

def calc_test_statistics (grades_1, grades_2):
    n12 = 0; n21 = 0
    for i in range(len(grades_1)) :
        n12 += grades_1[i] and not grades_2[i]
        n21 += not grades_1[i] and grades_2[i]
    n_star = n12 + n21
    z0 = (n21 - n12) / np.sqrt(n_star)
    p_value = binomtest(n21, n_star, alternative='greater').pvalue

    return n12, n21, n_star, z0, p_value

calc_test_statistics(grades_1, grades_2)

(1, 45, 46, 6.487446070815474, 6.679101716144942e-13)